In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$

\begin{pmatrix}
y_0 \\
y_1 \\
y_2 \\
\vdots \\
y_n
\end{pmatrix}

=

\begin{pmatrix}
x_{00} & x_{01}=1 \\
x_{10} & x_{11}=1 \\
x_{20} & x_{21}=1 \\
x_{30} & x_{31}=1 \\
\vdots & \vdots \\
x_{n0} & x_{n1}=1 \\
\end{pmatrix} 

\cdot

\begin{pmatrix}
w_0 \\
w_1
\end{pmatrix}

$$

$$ Y = X \cdot w $$
$$ X^T \cdot X \cdot \omega = X^T \cdot Y $$
$$ (X^T \cdot X)^{-1} \cdot X^T \cdot X \cdot \omega = (X^T \cdot X)^{-1} \cdot X^T \cdot Y $$
$$ \omega = (X^T \cdot X)^{-1} \cdot X^T \cdot Y $$


In [ ]:
import torch as tr

import import_ipynb
from common import assert_eq # type: ignore

def linear_regression_1d_ne(X: tr.Tensor, Y: tr.Tensor) -> tuple[float, float]:
    """
    Solves linear regression in one dimension using the normal equations.

    Args:
        X: Input features tensor of shape (size, 2), where the first column is the feature values 
            and the second column is a column of ones for the intercept term.
        Y: Target values tensor of shape (size, ).

    Returns:
        tuple: (slope, intercept)
    """
    
    size = X.shape[0]
    assert_eq(X.shape, (size, 2))
    assert_eq(Y.shape, (size, ))

    W = (X.t() @ X).inverse() @ X.t() @ Y
    assert_eq(W.shape, (2,))
    
    #
    # A preferable and more numerically stable solution that internally uses SVD decomposition.
    #
    # https://docs.pytorch.org/docs/stable/generated/torch.linalg.lstsq.html
    #
    # (W, residuals, rank, singular_values) = lstsq(X, Y)
    #

    slope = W[0]
    intercept = W[1]

    return (slope.item(), intercept.item())


def _test_linear_regression_1d_ne(a: float, b: float, s: int, m: float, n: float, atol=0.1) -> None:
    """
    Tests the linear regression normal equation solution by generating synthetic data with a known slope and intercept, 
    adding noise, and verifying that the computed slope and intercept are close to the original values.

    Args:
        a: Model's slope
        b: Model's intercept
        s: Samples
        m: Max value of x
        n: Noise level (%)
        atol: Absolute tolerance for comparison
    """

    N = n * (m / 100.0) * tr.randn((s, ), dtype=tr.float32)
    assert_eq(N.shape, (s, ))

    X = tr.arange(0, m, m/s, dtype=tr.float32)
    assert_eq(X.shape, (s, ))

    Y = (a * X + b) + N
    assert_eq(Y.shape, (s, ))

    X = tr.stack((X, tr.ones_like(X)), dim=1)
    assert_eq(X.shape, (s, 2))

    (slope, intercept) = linear_regression_1d_ne(X, Y)
    assert_eq(slope, a, atol)
    assert_eq(intercept, b, atol)


def test_linear_regression_1d_ne() -> None:
    _test_linear_regression_1d_ne(a=1.0, b=1.0, s=100, m=10.0, n=0.0)
    _test_linear_regression_1d_ne(a=2.0, b=3.0, s=100, m=10.0, n=0.0)
    _test_linear_regression_1d_ne(a=3.0, b=6.0, s=100, m=10.0, n=0.0)

    _test_linear_regression_1d_ne(a=-1.0, b=-1.0, s=100, m=10.0, n=0.0)
    _test_linear_regression_1d_ne(a=-2.0, b=-3.0, s=100, m=10.0, n=0.0)
    _test_linear_regression_1d_ne(a=-3.0, b=-6.0, s=100, m=10.0, n=0.0)

    _test_linear_regression_1d_ne(a=1.0, b=1.0, s=100, m=10.0, n=3.0, atol=0.2)
    _test_linear_regression_1d_ne(a=2.0, b=3.0, s=100, m=10.0, n=6.0, atol=0.5)
    _test_linear_regression_1d_ne(a=3.0, b=6.0, s=100, m=10.0, n=9.0, atol=1.0)

    _test_linear_regression_1d_ne(a=-1.0, b=-1.0, s=100, m=10.0, n=3.0, atol=0.2)
    _test_linear_regression_1d_ne(a=-2.0, b=-3.0, s=100, m=10.0, n=6.0, atol=0.5)
    _test_linear_regression_1d_ne(a=-3.0, b=-6.0, s=100, m=10.0, n=9.0, atol=1.0)


if __name__ == "__main__":
    test_linear_regression_1d_ne()
